## Feature Selection

### Variance Threshold


The variance of a feature is given by:

$$
\mathrm{Var}(X) = \frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2
$$

where:

- $x_i$ = $i^{th}$ observation
- $\bar{x}$ = mean of the feature
- $n$ = number of observations

A **low variance** means the feature values change very little across samples, while a **high variance** means the feature values are more spread out.

`VarianceThreshold` removes features whose variance is **less than or equal to a specified threshold**.

By default:

$$
\text{threshold} = 0
$$

So only **constant features** (variance = 0) are removed.

In [1]:
from sklearn.feature_selection import VarianceThreshold
import pandas as pd

In [2]:
df = pd.DataFrame({
    "A":[1,1,1,1],
    "B":[1,2,3,3],
    "C":[0,0,0,0]
})
df

,A,B,C
0,1,1,0
1,1,2,0
2,1,3,0
3,1,3,0


In [3]:
select = VarianceThreshold(threshold=0.5)
x = select.fit_transform(df)
x

array([[1],
       [2],
       [3],
       [3]])

In [4]:
feature = select.get_feature_names_out()
print(feature)

['B']


## SELECT KBest

In [5]:
from sklearn.feature_selection import SelectKBest

### 1.Scoring function - chi2

#### Chi-Square Formula

$$
\chi^2 = \sum \frac{(O - E)^2}{E}
$$

where

- $O$ = Observed frequency
- $E$ = Expected frequency

A larger $\chi^2$ value indicates a stronger association between the feature and the target.

In [6]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import chi2
import pandas as pd

In [8]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

selector = SelectKBest(score_func=chi2, k=2)

X_new = selector.fit_transform(X, y)

print(X.shape)
print(X_new.shape)

(150, 4)
(150, 2)


##  Scoring function - 2 - Mutual Information (MI)

Mutual Information measures **how much information a feature provides about the target**.

- If knowing the feature reduces uncertainty about the target, the Mutual Information is **high**.
- If the feature and target are independent, the Mutual Information is **0**.

Unlike correlation, Mutual Information can detect **both linear and non-linear relationships**.

### Formula

$$
I(X;Y)=\sum_{x \in X}\sum_{y \in Y}
P(x,y)\log\left(\frac{P(x,y)}{P(x)P(y)}\right)
$$

where:

- $P(x,y)$ = Joint probability of $X$ and $Y$
- $P(x)$ = Probability of $X$
- $P(y)$ = Probability of $Y$



In [10]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import mutual_info_classif
import pandas as pd


iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target


selector = SelectKBest(
    score_func=mutual_info_classif,
    k=2
)

X_new = selector.fit_transform(X, y)

print(X.shape)
print(X_new.shape)

(150, 4)
(150, 2)


##  Scoring function 3 - ANOVA F-Test (F-Statistic)

The ANOVA F-test measures **how well a feature separates different classes**.

It compares:

- **Between-group variance** → How far apart the class means are.
- **Within-group variance** → How spread out the samples are within each class.

### Formula

$$
F=\frac{\text{Between-group variance}}
{\text{Within-group variance}}
$$

or

$$
F=\frac{MS_{\text{between}}}{MS_{\text{within}}}
$$

where:

- $MS_{\text{between}}$ = Mean Square Between groups
- $MS_{\text{within}}$ = Mean Square Within groups

### Interpretation

- **High F-score**
  - Class means are far apart.
  - Samples within each class are tightly clustered.
  - Feature is useful for classification.

- **Low F-score**
  - Class means are close together.
  - Classes overlap significantly.
  - Feature is less useful.

### Advantages

- Fast and computationally efficient.
- Works well for continuous numerical features.
- Does **not** require non-negative feature values.

### Disadvantages

- Assumes a linear relationship between the feature and the target.
- Less effective for complex non-linear relationships.

### When to Use

- **Classification:** `f_classif`
- **Regression:** `f_regression`

In [11]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import f_classif
import pandas as pd


iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target


selector = SelectKBest(
    score_func=f_classif,
    k=2
)

X_new = selector.fit_transform(X, y)

print(X.shape)
print(X_new.shape)

(150, 4)
(150, 2)


# SelectPercentile

`SelectPercentile` is a feature selection method that selects the **top percentage of features** based on a scoring function.

Unlike `SelectKBest`, which selects a fixed number of features (`k`), `SelectPercentile` selects a specified **percentage** of the highest-scoring features.

## Syntax

```python
from sklearn.feature_selection import SelectPercentile

selector = SelectPercentile(
    score_func=f_classif,
    percentile=20
)
```

## Parameters

- `score_func` → Function used to score each feature.
- `percentile` → Percentage of top features to keep.

## Supported Scoring Functions

### Classification
- `chi2`
- `f_classif`
- `mutual_info_classif`

### Regression
- `f_regression`
- `mutual_info_regression`

## Difference from SelectKBest

- **SelectKBest** → Keeps the top **k** features.
- **SelectPercentile** → Keeps the top **percentage** of features.

Both use the **same scoring functions**; only the selection criterion is different.

 # GenericUnivariateSelect

`GenericUnivariateSelect` is a flexible feature selection method that supports multiple feature selection strategies within a single class.

It uses the **same scoring functions** as `SelectKBest` and `SelectPercentile`. The only difference is that you specify the selection strategy using the `mode` parameter.

## Syntax

```python
from sklearn.feature_selection import GenericUnivariateSelect

selector = GenericUnivariateSelect(
    score_func=f_classif,
    mode="k_best",
    param=5
)
```